# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset schema is provided via the following Croissant JSON-LD URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading

Below, we load the dataset metadata using `mlcroissant` and print its key metadata attributes.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
ds = mlc.Dataset(croissant_url)

# Access top-level dataset metadata attributes
metadata = ds.metadata

print(f"\033[1m{metadata.name}\033[0m")
print(metadata.description)
print(f"\nIdentifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")

## 2. Data Overview

Explore available record sets and their fields. Each record set and field is referenced by its unique `@id` as required by the Croissant data model.

A record set typically corresponds to a table in the dataset. Fields and columns within each record set are uniquely identified by their `@id`.

In [ ]:
# List all available record sets with @id

record_sets = list(ds.schema.record_sets)
print("Available record sets:")
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")

if not record_sets:
    print("(No record sets were detected in schema. Run 'ds.records()' directly to probe data records.)")

# For datasets with record sets, list fields in each
for rs in record_sets:
    print(f"\nFields in record set @id: {rs['@id']}")
    if 'field' in rs:
        # field can be a list or a dict (single field)
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for f in fields:
            if isinstance(f, dict):
                print(f"  - @id: {f['@id']} name: {f.get('name', '(no name)')}")
            else:
                print(f"  - @id: {f}")
    else:
        print("  (No fields declared in this record set)")


## 3. Data Extraction

Load data from each record set using the correct record set `@id`. Data can be loaded as Pandas DataFrames for downstream analysis. All references to record sets and fields use their `@id`.

In [ ]:
# Identify available record_set @id(s)
record_set_ids = [rs['@id'] for rs in ds.schema.record_sets]
if not record_set_ids:
    # Try to use default record set if record_sets is empty (usually for a single table dataset)
    print("No explicit record sets in the schema. Attempting to load all records via ds.records()...")
    all_records = list(ds.records())
    df = pd.DataFrame(all_records)
    print("DataFrame columns:", df.columns.tolist())
    display(df.head())
    # For subsequent steps, we simulate a record_set with '@id': 'main' (used as variable)
    dataframes = {'main': df}
    used_record_set_id = 'main'
else:
    dataframes = {}
    for record_set_id in record_set_ids:
        records = list(ds.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrames for record sets: {list(dataframes.keys())}")
    # Pick the first record_set for example EDA
    used_record_set_id = record_set_ids[0]

# Show an overview of columns in the chosen record set
print(f"\nColumns in record set '{used_record_set_id}':")
print(dataframes[used_record_set_id].columns.tolist())

dataframes[used_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Common data processing operations: filter records based on numeric field values, normalize the field, and group data by a categorical attribute.

All references to fields use their Croissant `@id`. Replace field names with `@id`s for unique referencing, as required.

In [ ]:
# To proceed: Select a numeric field (by @id) and a group field (@id) from the DataFrame's columns

df = dataframes[used_record_set_id]

# Print column names (likely field @id's) for reference
print("DataFrame columns (fields, possibly @id):")
print(df.columns.tolist())

# Try to automatically select a numeric field (usually age or similar)
numeric_field_candidates = [c for c in df.columns if df[c].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[c])]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No numeric field detected for EDA! Proceeding with dummy numeric field.")
    numeric_field_id = df.columns[0]

# Set a filter threshold for analysis
threshold = 50

# Filter records where numeric_field > threshold
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered {len(filtered_df)} records with {numeric_field_id} > {threshold}.")
print(filtered_df.head())

# Normalize the numeric field (Z-score)
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized field '{numeric_field_id}':")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a group field (pick first categorical/text field or the second column)
categorical_fields = [c for c in df.columns if pd.api.types.is_object_dtype(df[c]) and not df[c].equals(df[numeric_field_id])]
if categorical_fields:
    group_field_id = categorical_fields[0]
    print(f"\nGrouping filtered records by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
    print(grouped_df.head())
else:
    print("\nNo clear categorical group field available for grouping.")

## 5. Visualization

Visualize distributions of the selected numeric field and relationships with a grouping attribute (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (before filtering and normalization)
plt.figure(figsize=(6, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If grouping field is available, show boxplot/grouped means
if 'group_field_id' in locals() and group_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

This notebook demonstrated metadata loading and record extraction from the FAIR² colorectal cancer survivor dataset using `mlcroissant`. We explored the data structure by `@id`, performed basic filtering, normalization, grouping, and simple visualizations on a selected numeric field. 

For further work, you can proceed with more advanced modeling, statistical analysis, or integrate other Croissant-powered datasets.